# 01 · Data and image → embedding pipeline (Milestone 1)

This notebook covers Milestone 1:

1. Download COCO Captions val2017 and clean the 5,000 images and ~25,000 captions
2. Split by image into train / val / test (4,000 / 500 / 500)
3. Tokenize captions with the GPT-2 tokenizer
4. Encode every image with a frozen CLIP ViT-B/32 (pooled 512-d vector + 50×768 patch features)
5. Five image : caption test runs through the whole pipeline

Set `LIMIT = 64` for a quick local check; use `None` for the full dataset.

In [ ]:
# Run from the repo root (works whether Jupyter starts in notebooks/ or the root).
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

# On Colab, run once instead:
# !git clone https://github.com/Yiting0210/clip-gpt2-captioning.git
# %cd clip-gpt2-captioning
# !pip install -q -r requirements.txt

import json, torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| CUDA", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
LIMIT = None   # e.g. 64 for a quick test on a laptop
from src import data
data.download(LIMIT)          # ~240 MB annotations + ~780 MB images (or LIMIT single images)
report = data.prepare(LIMIT)  # clean captions, verify images, split

## Cleaning report

What the cleaning step removed or converted. Non-RGB images are kept and converted to RGB before CLIP.

In [ ]:
import pandas as pd
pd.Series(report, name='count').to_frame()

## GPT-2 tokenization

Captions get a leading space (GPT-2 merges it into the first word) and end with `<|endoftext|>`. The model input is `[bos] + caption + [eos]`; GPT-2 uses `<|endoftext|>` for both.

In [ ]:
from transformers import GPT2TokenizerFast
tok = GPT2TokenizerFast.from_pretrained('gpt2')
caps, splits = data.load_captions(), data.load_splits()
example = next(iter(caps.values()))['captions'][0]
ids = tok.encode(' ' + example)
print(example)
print(ids)
print([tok.decode([i]) for i in ids])
{k: len(v) for k, v in splits.items()}

In [ ]:
import matplotlib.pyplot as plt
# Fixed categorical order (validated reference palette); recessive grid, no top/right spines.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=SERIES), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.color": "#e4e3df",
                     "grid.linewidth": 0.8, "axes.edgecolor": "#9a9893", "axes.labelcolor": "#52514e",
                     "xtick.color": "#52514e", "ytick.color": "#52514e", "lines.linewidth": 2,
                     "figure.dpi": 110, "font.size": 10})

lengths = [len(tok.encode(' ' + c)) for r in caps.values() for c in r['captions']]
stats = data.caption_length_stats(caps, tok, max_tokens=40)
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(lengths, bins=range(0, max(lengths) + 2), color=SERIES[0], edgecolor='white', linewidth=1)
ax.axvline(40, color='#52514e', linestyle='--', linewidth=1)
ax.text(40.5, ax.get_ylim()[1] * 0.9, 'max_caption_tokens = 40', color='#52514e', fontsize=9)
ax.set(title='Most captions are 10–15 GPT-2 tokens long', xlabel='tokens per caption', ylabel='captions')
plt.show(); stats

## CLIP image embeddings

Computed once and cached in `data/processed/clip_clip-vit-base-patch32.pt`. Training never reads images again, which is what makes each experiment fast.

In [ ]:
CLIP = 'openai/clip-vit-base-patch32'
emb = data.load_embeddings(CLIP) if data.embedding_path(CLIP).exists() else None
if emb is None or set(emb['image_ids']) != set(caps):   # missing, or left over from a smaller LIMIT
    data.embed(CLIP, device=DEVICE)
    emb = data.load_embeddings(CLIP)
print('pooled', tuple(emb['pooled'].shape), '| patch', tuple(emb['patch'].shape))

Sanity check: an image's nearest neighbour in CLIP space should show similar content.

In [ ]:
import torch.nn.functional as F
P = F.normalize(emb['pooled'].float(), dim=-1)
sims = P @ P.T
sims.fill_diagonal_(-1)
print('mean cosine similarity between different images:', round(float(sims[sims > -1].mean()), 3))
q = emb['image_ids'][0]; nn = emb['image_ids'][int(sims[0].argmax())]
print('query :', caps[q]['captions'][0]); print('neighbour:', caps[nn]['captions'][0])

## Five image : caption test runs

Five test images through the whole pipeline. Each image shows:

- **REF**: a human reference caption
- **retrieval**: the caption of the most similar *training* image in CLIP space (no training needed)
- **untrained**: GPT-2 conditioned on a randomly initialised prefix. It is not meant to be good; it proves image → CLIP → mapping → GPT-2 → text runs end to end before training starts in Milestone 2.

In [ ]:
from IPython.display import Image as Show
from src.evaluate import retrieval_captions, save_gallery
from src.model import ClipCaptionModel, load_tokenizer
from src.decode import greedy

ids5 = splits['test'][:5]
retr = retrieval_captions(ids5, emb, caps, splits)
torch.manual_seed(0)
untrained = ClipCaptionModel('gpt2', 'prefix', 'prefix', feat_dim=512).eval()
feats = emb['pooled'][[emb['index'][i] for i in ids5]].float()
raw = dict(zip(ids5, greedy(untrained, feats, load_tokenizer(), max_len=20)))
for i in ids5:
    print(f'{i}\n  REF      : {caps[i]["captions"][0]}\n  retrieval: {retr[i]}\n  untrained: {raw[i]}')
Show(str(save_gallery(ids5, {'retrieval': retr, 'untrained': raw}, caps, Path('results/m1_test_runs.png'))))